In [196]:
#installing FinRL library
!pip install wrds
!pip install swig

!apt-get update -y -qq && apt-get install -y -qq cmake libopenmpi-dev python3-dev zlib1g-dev libgl1-mesa-glx swig
!pip install git+https://github.com/AI4Finance-Foundation/FinRL.git

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: C:\Users\nikhi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: C:\Users\nikhi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
'apt-get' is not recognized as an internal or external command,
operable program or batch file.


Defaulting to user installation because normal site-packages is not writeable
  Cloning https://github.com/AI4Finance-Foundation/FinRL.git to c:\users\nikhi\appdata\local\temp\pip-req-build-upae03oc
  Resolved https://github.com/AI4Finance-Foundation/FinRL.git to commit 8cf3cacc6f570d26b430e403ea522c8fe9e6876a
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Cloning https://github.com/AI4Finance-Foundation/ElegantRL.git to c:\users\nikhi\appdata\local\temp\pip-install-a4hua_x6\elegantrl_db351e7455104bb79f0dba28c9eced0c
  Resolved https://github.com/AI4Finance-Foundation/ElegantRL.git to commit 3bdc958c8e624b61a9e661832b01fef816924f61
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): f

  Running command git clone --filter=blob:none --quiet https://github.com/AI4Finance-Foundation/FinRL.git 'C:\Users\nikhi\AppData\Local\Temp\pip-req-build-upae03oc'
  Running command git clone --filter=blob:none --quiet https://github.com/AI4Finance-Foundation/ElegantRL.git 'C:\Users\nikhi\AppData\Local\Temp\pip-install-a4hua_x6\elegantrl_db351e7455104bb79f0dba28c9eced0c'
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> [18 lines of output]
      C:\Users\nikhi\AppData\Local\Temp\pip-install-a4hua_x6\pyfolio_3600ed936dd046ec96e3bf4100286145\versioneer.py:468: SyntaxWarning: invalid escape sequence '\s'
        LONG_VERSION_PY['git'] = '''
      Traceback (most recent call last):
        File "<string>", line 2, in <module>
        File "<pip-setuptools-caller>", line 34, in <module>
        File "C:\Users\nikhi\AppData\Local\Temp\pip-install-a4hua_x6\pyfolio_3600ed936dd046ec96e3bf4100286145\setup.py", line 71, in <mo

In [197]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import datetime
import gym
from finrl import config
from finrl.agents.stablebaselines3.models import DRLEnsembleAgent
from finrl.meta.preprocessor.preprocessors import FeatureEngineer
from pprint import pprint
import sys
sys.path.append("../FinRL-Library")
import itertools

In [198]:
import warnings
warnings.filterwarnings("ignore")

In [199]:
import os
from finrl.main import check_and_make_directories
from finrl.config import (
    DATA_SAVE_DIR,
    TRAINED_MODEL_DIR,
    TENSORBOARD_LOG_DIR,
    RESULTS_DIR,
    INDICATORS,
    TRAIN_START_DATE,
    TRAIN_END_DATE,
    TEST_START_DATE,
    TEST_END_DATE,
    TRADE_START_DATE,
    TRADE_END_DATE,
)

import os
if not os.path.exists("./" + config.DATA_SAVE_DIR):
    os.makedirs("./" + config.DATA_SAVE_DIR)
if not os.path.exists("./" + config.TRAINED_MODEL_DIR):
    os.makedirs("./" + config.TRAINED_MODEL_DIR)
if not os.path.exists("./" + config.TENSORBOARD_LOG_DIR):
    os.makedirs("./" + config.TENSORBOARD_LOG_DIR)
if not os.path.exists("./" + config.RESULTS_DIR):
    os.makedirs("./" + config.RESULTS_DIR)

In [200]:
TRAIN_START_DATE = '2017-01-01'
TRAIN_END_DATE = '2021-10-01'
TEST_START_DATE = '2021-10-01'
TEST_END_DATE = '2023-03-01'

In [201]:
df = pd.read_csv('C:/Users/nikhi/Desktop/Capstone project/historical_data.csv')


In [202]:
df.head(5)

,date,close,high,low,open,volume,tic,day
0,2004-01-01,0.105000,0.105000,0.105000,0.105000,0,DYA.TO,3
1,2004-01-01,1.378000,1.378000,1.378000,1.378000,0,EDT.TO,3
2,2004-01-01,0.300000,0.300000,0.300000,0.300000,0,TSK.TO,3
3,2004-01-02,0.489625,0.489625,0.489625,0.489625,0,AAB.TO,4
4,2004-01-02,22.070263,22.211644,21.817266,21.899118,413000,ABX.TO,4


In [203]:
INDICATORS = ['macd',
               'rsi_30',
               'cci_30',
               'dx_30']

In [204]:
TICKERS = ["RY.TO", "TD.TO", "BNS.TO", "ENB.TO", "SHOP.TO"] 

In [205]:
df = df[df['tic'].isin(TICKERS)]


In [206]:
df.shape

(23500, 8)

In [207]:
feature_engineer = FeatureEngineer(use_technical_indicator=True, tech_indicator_list=INDICATORS)


In [208]:
df = feature_engineer.add_technical_indicator(df)

In [209]:
df = feature_engineer.add_turbulence(df)

In [210]:
df["date"] = pd.to_datetime(df["date"]) 

In [211]:
stock_dimension = len(df.tic.unique())
state_space = 1 + 2*stock_dimension + len(INDICATORS)*stock_dimension
print(f"Stock Dimension: {stock_dimension}, State Space: {state_space}")

Stock Dimension: 5, State Space: 31


In [212]:
df.head(5)

,date,close,high,low,open,volume,tic,day,macd,rsi_30,cci_30,dx_30,turbulence
0,2004-01-02,3.425969,3.436876,3.397921,3.406751,1108800,BNS.TO,4,0.00000,NaN,NaN,NaN,0.0
1,2004-01-02,5.307566,5.310516,5.251511,5.251511,439600,ENB.TO,4,0.00000,NaN,NaN,NaN,0.0
2,2004-01-02,13.910172,14.113713,13.834123,13.834123,1938000,RY.TO,4,0.00000,NaN,NaN,NaN,0.0
3,2004-01-02,9.893545,9.945557,9.800829,9.825704,1756600,TD.TO,4,0.00000,NaN,NaN,NaN,0.0
4,2004-01-05,3.422853,3.436357,3.420256,3.420775,1675400,BNS.TO,0,-0.00007,0.0,66.666667,NaN,0.0


In [213]:
env_kwargs = {
    "hmax": 100,
    "initial_amount": 1000000,
    "buy_cost_pct": 0.001,
    "sell_cost_pct": 0.001,
    "state_space": state_space,
    "stock_dim": stock_dimension,
    "tech_indicator_list": INDICATORS,
    "action_space": stock_dimension,
    "reward_scaling": 1e-4,
    "print_verbosity":5

}


In [214]:
rebalance_window = 63 # rebalance_window is the number of days to retrain the model
validation_window = 63 # validation_window is the number of days to do validation and trading (e.g. if validation_window=63, then both validation and trading period will be 63 days)

ensemble_agent = DRLEnsembleAgent(df,
                 train_period=(TRAIN_START_DATE,TRAIN_END_DATE),
                 val_test_period=(TEST_START_DATE,TEST_END_DATE),
                 rebalance_window=rebalance_window,
                 validation_window=validation_window,
                 **env_kwargs)


In [215]:
from stable_baselines3 import SAC

SAC_model_kwargs = {
    "buffer_size": 100000,
    "learning_rate": 0.0005,
    "batch_size": 64,
    "ent_coef": "auto",  # Automatic entropy coefficient
    "tau": 0.005,       # Target network update rate
    "gamma": 0.99,      # Discount factor
    "learning_starts": 1000
}


timesteps_dict = {'a2c' : 0,
                 'ppo' : 0,
                 'ddpg' : 0,
                 'sac' : 10_000,  # Focus on SAC
                 'td3' : 0
                 }

In [216]:
df_summary = ensemble_agent.run_ensemble_strategy(
    A2C_model_kwargs={},  
    PPO_model_kwargs={},  
    DDPG_model_kwargs={},  
    SAC_model_kwargs=SAC_model_kwargs,  # Use SAC config
    TD3_model_kwargs={},  
    timesteps_dict=timesteps_dict
)

============Start Ensemble Strategy============
turbulence_threshold:  60.18625422507027
======Model training from:  2017-01-01 to  2021-10-04 00:00:00
======a2c Training========
{}
Using cpu device
Logging to tensorboard_log/a2c\a2c_126_2


======a2c Validation from:  2021-10-04 00:00:00 to  2022-01-05 00:00:00
a2c Sharpe Ratio:  -0.16962801171200861
======ddpg Training========
{}
Using cpu device
Logging to tensorboard_log/ddpg\ddpg_126_2
======ddpg Validation from:  2021-10-04 00:00:00 to  2022-01-05 00:00:00
ddpg Sharpe Ratio:  0.242884150618702
======td3 Training========
{}
Using cpu device
Logging to tensorboard_log/td3\td3_126_2
======td3 Validation from:  2021-10-04 00:00:00 to  2022-01-05 00:00:00
td3 Sharpe Ratio:  0.0
======sac Training========
{'buffer_size': 100000, 'learning_rate': 0.0005, 'batch_size': 64, 'ent_coef': 'auto', 'tau': 0.005, 'gamma': 0.99, 'learning_starts': 1000}
Using cpu device
Logging to tensorboard_log/sac\sac_126_2
day: 1192, episode: 5
begin_total_asset: 1000000.00
end_total_asset: 1491797.74
total_reward: 491797.74
total_cost: 999.00
total_trades: 5790
Sharpe: 0.512
----------------------------------
| time/              |           |
|    episodes        | 4         |
|    fps        

In [218]:
df_summary

,Iter,Val Start,Val End,Model Used,A2C Sharpe,PPO Sharpe,DDPG Sharpe,SAC Sharpe,TD3 Sharpe
0,126,2021-10-04 00:00:00,2022-01-05 00:00:00,SAC,-0.169628,0.318051,0.242884,0.392999,0.0
1,189,2022-01-05 00:00:00,2022-04-05 00:00:00,TD3,-0.260327,-0.056165,-0.100982,-0.159215,-0.029529
2,252,2022-04-05 00:00:00,2022-07-06 00:00:00,TD3,-0.551517,-0.509224,-0.436233,-0.327658,-0.153446
3,315,2022-07-06 00:00:00,2022-10-05 00:00:00,TD3,-0.017902,-0.095529,-0.133124,0.100025,0.151308


For SAC (a reinforcement learning model), tuning involves adjusting hyperparameters such as:
Learning rate – How fast the model updates weights.
Gamma (γ) – Discount factor for future rewards.
Batch size – Number of experiences used per training step.
Tau (τ) – Controls soft update of target networks.
Alpha (α) – Controls exploration vs. exploitation trade-off.

In [219]:
SAc_model_kwargs_TUNING = {
    "learning_rate": 0.0005,
    "buffer_size": 50000,
    "batch_size": 64,
    "ent_coef": "auto",  # Automatic entropy coefficient
    "tau": 0.005,       # Target network update rate
    "gamma": 0.99,      # Discount factor
    "learning_starts": 1000
}

In [220]:
df_summary_tuning = ensemble_agent.run_ensemble_strategy(
    A2C_model_kwargs={},  
    PPO_model_kwargs={},  
    DDPG_model_kwargs={},  
    SAC_model_kwargs=SAc_model_kwargs_TUNING,  # Use SAC config
    TD3_model_kwargs={},  
    timesteps_dict=timesteps_dict
)
df_summary_tuning

============Start Ensemble Strategy============
turbulence_threshold:  60.18625422507027
======Model training from:  2017-01-01 to  2021-10-04 00:00:00
======a2c Training========
{}
Using cpu device
Logging to tensorboard_log/a2c\a2c_126_3
======a2c Validation from:  2021-10-04 00:00:00 to  2022-01-05 00:00:00
a2c Sharpe Ratio:  0.22899223256664197
======ddpg Training========
{}
Using cpu device
Logging to tensorboard_log/ddpg\ddpg_126_3
======ddpg Validation from:  2021-10-04 00:00:00 to  2022-01-05 00:00:00
ddpg Sharpe Ratio:  0.2996524179757144
======td3 Training========
{}
Using cpu device
Logging to tensorboard_log/td3\td3_126_3
======td3 Validation from:  2021-10-04 00:00:00 to  2022-01-05 00:00:00
td3 Sharpe Ratio:  -0.177465909633174
======sac Training========
{'learning_rate': 0.0005, 'buffer_size': 50000, 'batch_size': 64, 'ent_coef': 'auto', 'tau': 0.005, 'gamma': 0.99, 'learning_starts': 1000}
Using cpu device
Logging to tensorboard_log/sac\sac_126_3
day: 1192, episode: 5
b

,Iter,Val Start,Val End,Model Used,A2C Sharpe,PPO Sharpe,DDPG Sharpe,SAC Sharpe,TD3 Sharpe
0,126,2021-10-04 00:00:00,2022-01-05 00:00:00,DDPG,0.228992,0.047356,0.299652,-0.148741,-0.177466
1,189,2022-01-05 00:00:00,2022-04-05 00:00:00,A2C,0.036849,-0.11642,-0.162717,-0.058881,-0.066191
2,252,2022-04-05 00:00:00,2022-07-06 00:00:00,SAC,-0.488831,-0.420925,-0.324423,-0.317728,-0.375589
3,315,2022-07-06 00:00:00,2022-10-05 00:00:00,TD3,-0.166076,-0.040575,-0.050484,0.049355,0.100025


SAC Hyperparameter Tuning with Optuna

In [175]:
!pip install optuna

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: C:\Users\nikhi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [191]:
import optuna
import gymnasium as gym 
import torch
from stable_baselines3 import SAC
from stable_baselines3.common.evaluation import evaluate_policy

In [177]:
# Define the environment
ENV_NAME = "Pendulum-v1"  # Change this to your custom environment if needed


In [ ]:
def optimize_sac(trial):
    """ Objective function for Optuna to optimize SAC hyperparameters. """
    
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-5, 3e-4)
    gamma = trial.suggest_float("gamma", 0.90, 0.99)
    batch_size = trial.suggest_categorical("batch_size", [64, 128, 256, 512])
    tau = trial.suggest_float("tau", 0.005, 0.05)
    alpha = trial.suggest_float("alpha", 0.05, 0.3)

In [192]:
def optimize_sac(trial):
    """ Objective function for Optuna to optimize SAC hyperparameters. """
    
    # Sample hyperparameters
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-5, 3e-4)
    gamma = trial.suggest_float("gamma", 0.90, 0.99)
    batch_size = trial.suggest_categorical("batch_size", [64, 128, 256, 512])
    tau = trial.suggest_float("tau", 0.005, 0.05)
    alpha = trial.suggest_float("alpha", 0.05, 0.3)
    
    # Create environment
    env = gym.make(ENV_NAME)

    # Define the SAC model **inside** the function
    model = SAC(
        "MlpPolicy",
        env,
        learning_rate=learning_rate,
        gamma=gamma,
        batch_size=batch_size,
        tau=tau,
        ent_coef=alpha,
        verbose=0,
        tensorboard_log="./sac_tuning/"
    )

    # Train model for a few time steps
    model.learn(total_timesteps=10000)

    # Evaluate performance
    mean_reward, _ = evaluate_policy(model, env, n_eval_episodes=5)

    env.close()
    
    return mean_reward

In [190]:
pip install gymnasium[box2d] stable-baselines3 optuna


Defaulting to user installation because normal site-packages is not writeable
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ---------------------------------------- 0.0/10.6 MB ? eta -:--:--
   -- ------------------------------------- 0.8/10.6 MB 4.8 MB/s eta 0:00:03
   ------ --------------------------------- 1.8/10.6 MB 5.0 MB/s eta 0:00:02
   ---------- ----------------------------- 2.9/10.6 MB 5.1 MB/s eta 0:00:02
   ---------------- ----------------------- 4.5/10.6 MB 5.8 MB/s eta 0:00:02
   --------------------- ------------------ 5.8/10.6 MB 6.0 MB/s eta 0:00:01
   --------------------------- ------------ 7.3/10.6 MB 6.3 MB/s eta 0:00:01
   ----------------------------------- ---- 9.4/10.6 MB 6.9 MB/s eta 0:00:01
   ---------------------------------------- 10.6/10.6 MB 6.8 MB/s eta 0:00:00
  Running setup.py clean for box2d-py
Failed to build box2d-py
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> [28 lines of output]
      Using setuptools (version 71.1.0).
      running bdist_wheel
      running build
      running build_py
      creating build
      creating build\lib.win-amd64-cpython-312
      creating build\lib.win-amd64-cpython-312\Box2D
      copying library\Box2D\Box2D.py -> build\lib.win-amd64-cpython-312\Box2D
      copying library\Box2D\__init__.py -> build\lib.win-amd64-cpython-312\Box2D
      creating build\lib.win-amd64-cpython-312\Box2D\b2
      copying library\Box2D\b2\__init__.py -> build\lib.win-amd64-cpython-312\Box2D\b2
      running build_ext
      building 'Box2D._Box2D' extension
      swigging Box2D\Box2D.i to Box2D\Box2D_wrap.cpp
      swig.exe -python -c++ -IBox2D -small -O -includeall -ignoremissing -w201 -globals b2Globals -outdir library\Box2D -keyword -w511 -D_SWIG_KWARGS -o Box2D\Box2D_wrap.cpp Box2D\Box2D.i
      Box2D\Common\

In [193]:
# Run Optuna optimization
study = optuna.create_study(direction="maximize")  # Maximize the reward
study.optimize(optimize_sac, n_trials=20)  # Run 20 trials


[I 2025-03-03 03:51:49,649] A new study created in memory with name: no-name-58ade97c-b072-4235-9abc-8dcfa2f43abc
[I 2025-03-03 03:56:50,047] Trial 0 finished with value: -1031.1671267524362 and parameters: {'learning_rate': 5.062425619318205e-05, 'gamma': 0.9762215219235065, 'batch_size': 512, 'tau': 0.043324338558708965, 'alpha': 0.11054429661089386}. Best is trial 0 with value: -1031.1671267524362.
[I 2025-03-03 03:59:55,870] Trial 1 finished with value: -1677.5736792325974 and parameters: {'learning_rate': 1.0337890818149418e-05, 'gamma': 0.979757912042467, 'batch_size': 128, 'tau': 0.014850779447869621, 'alpha': 0.15739903735542082}. Best is trial 0 with value: -1031.1671267524362.
[I 2025-03-03 04:03:07,397] Trial 2 finished with value: -792.9772980809212 and parameters: {'learning_rate': 9.882759814175665e-05, 'gamma': 0.9220796892184415, 'batch_size': 256, 'tau': 0.005465935002723749, 'alpha': 0.09923592660302245}. Best is trial 2 with value: -792.9772980809212.
[I 2025-03-03 0

In [194]:
# Print best parameters
print("Best hyperparameters:", study.best_params)

Best hyperparameters: {'learning_rate': 0.00021319194087739252, 'gamma': 0.9899375818292613, 'batch_size': 128, 'tau': 0.025405564719087035, 'alpha': 0.2515576894516406}


references:
[GitHUB](https://github.com/AI4Finance-Foundation/FinRL/blob/master/examples/Stock_NeurIPS2018_SB3.ipynb)
https://stable-baselines3.readthedocs.io/en/master/guide/rl_tips.html#optimizing-hyperparameters-with-optuna
https://stable-baselines3.readthedocs.io/en/master/guide/rl_tips.html#optimizing-hyperparameters-with-optuna    